# MLDU-E — Pythia-70M PGA (standalone)

Self-contained notebook for the Pythia-70M PGA experiment, extracted from `mldu-e-pythia-followup-kaggle_pipleline3.ipynb` (cells 92–107).

**What this notebook does:**
1. Installs `peft + transformers + accelerate` and pins datasets versions if needed.
2. Loads Pythia-70M and auto-builds memorized/clean text pools (idempotent — caches to `pythia_memorized.json` / `pythia_clean.json`).
3. Trains a LoRA-PGA adapter on attention + MLP layers for 200 epochs with the deterministic seed-reset patch.
4. Saves the LoRA adapter and writes per-layer probe accuracies to `mldu_e_pga_pythia70m.json`.
5. Runs the held-out adversarial-probe robustness sweep (six variants) and saves `mldu_e_pga_pythia70m_robustness.json`.
6. Generates the robustness figure.

**Reproducibility:** Cell 102 of the source notebook had no per-cell seed reset — random state was advanced unpredictably by upstream cells (model load, data-pool auto-build, sklearn probe fit) before LoRA initialisation. This standalone version applies a fresh seed reset immediately before LoRA training, so re-runs from the same kernel give bit-identical post-PGA per-layer values (modulo CUDA non-determinism).

**Expected output:** `post_per_layer = {0: 0.643, 1: 0.714, 2: 0.571, 3: 0.286, 4: 0.429, 5: 0.286, 6: 0.143}`. If your run gives `0.071` at layers 5–6 instead, that's the older, no-longer-reproducible value from the paper; see the README in the github submission for the discussion.


## 0. Install + setup

In [9]:
!pip install -q peft transformers accelerate

In [10]:
import os, json, time, math, random, copy, warnings
import numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive/MIDU')
except (ImportError, Exception):
    DRIVE = Path('./MIDU'); DRIVE.mkdir(parents=True, exist_ok=True)
    print(f'(non-Colab) DRIVE = ' + str(DRIVE))
ART   = DRIVE / 'MLDU_E' / 'artifacts'; ART.mkdir(parents=True, exist_ok=True)
LORA_DIR = DRIVE / 'pythia70m_pga_lora'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f'device: {DEVICE}')

(non-Colab) DRIVE = MIDU
device: cuda


## 1. Hyperparameters

In [11]:
MODEL_NAME    = 'EleutherAI/pythia-70m'
PROBE_LAYER   = 4                    # peak-gap layer per parent paper
N_HIDDEN      = 7                    # embed + 6 transformer layers
MAX_LEN       = 128
LORA_R        = 16
LR            = 1e-4
EPOCHS        = 200
REFIT_EVERY   = 25
LAMBDA_ALIGN  = 1.0
LAMBDA_CE     = 1.0
ALIGN_LAYERS  = list(range(1, 7))     # post-embed through final
PROBE_C       = 1.0                   # used during PGA training (the 'trained-against' probe)

## 2. Load Pythia-70M + memorized/clean data

In [12]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE)
print(f'Loaded {MODEL_NAME}: {sum(p.numel() for p in base.parameters()):,} params')

# --- AUTO-BUILD: create pythia_memorized.json/pythia_clean.json if missing ---
if not (DRIVE / 'pythia_memorized.json').exists() or not (DRIVE / 'pythia_clean.json').exists():
    print('pythia_memorized.json/pythia_clean.json missing -- running build step (idempotent, ~1 min on T4)...')
    import torch.nn.functional as _F
    _CANDIDATES = [
        "Copyright (c) 2016 The Linux Foundation. Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the \"Software\"), to deal in the Software without restriction, including without limitation the rights",
        "GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C) 1989, 1991 Free Software Foundation, Inc., 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA Everyone is permitted to copy and distribute verbatim copies of this license document,",
        "Licensed under the Apache License, Version 2.0 (the \"License\"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law",
        "The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog.",
        "Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat.",
        "Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met: 1. Redistributions of source code must retain the above copyright notice, this list of conditions",
        "This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.",
        "ABOVE COPYRIGHT NOTICE AND THIS PERMISSION NOTICE SHALL BE INCLUDED IN ALL COPIES OR SUBSTANTIAL PORTIONS OF THE SOFTWARE. THE SOFTWARE IS PROVIDED \"AS IS\", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED",
        "We gratefully acknowledge support from the Simons Foundation and member institutions. Help | Advanced Search All fields Title Author Abstract Comments Journal reference ACM classification MSC classification",
        "Skip to main content Skip to search Help Advanced Search | CODES: All Title Author Abstract Cite search results export to BibTeX export as Text export as PDF Submit Search Home Browse Latest",
    ]
    _CLEAN_POOL = [
        "The research team conducted experiments with three different sample groups to evaluate the effectiveness of the new treatment protocol. Each group received a different dosage level based on the established clinical guidelines.",
        "Environmental monitoring stations across the region recorded significant changes in atmospheric composition over the past decade. Scientists attribute these variations to multiple factors including industrial activity and seasonal weather patterns.",
        "The committee reviewed the submitted proposals according to the evaluation criteria outlined in the original request for applications. Each proposal was scored independently by three reviewers using the standard scoring rubric.",
        "Participants were recruited through local community centers and provided informed consent before beginning the study. The research protocol was approved by the institutional review board in accordance with ethical guidelines.",
        "The quarterly financial report indicated moderate growth in revenue despite challenging market conditions. Management attributes this performance to strategic investments in product development and expansion.",
        "Field observations at the coastal ecosystem site revealed several previously undocumented species. Researchers collected samples for further taxonomic analysis and genetic sequencing at the regional biodiversity center.",
        "The manuscript presents findings from a longitudinal study tracking educational outcomes across twelve school districts. Statistical analysis revealed significant correlations between early intervention programs and later academic performance.",
        "Archaeological excavations at the northern site uncovered artifacts dating from multiple historical periods. The team documented each find using standard methodology and prepared detailed reports for the cultural heritage archive.",
        "Quarterly earnings exceeded analyst expectations by a substantial margin due to stronger than projected consumer demand. The chief executive officer credited the performance to successful product launches in emerging markets.",
        "The conference proceedings include papers on a wide range of topics in computational science. Plenary sessions featured keynote presentations by leading researchers from universities and industry laboratories.",
    ]
    @torch.no_grad()
    def _logp(text, n_pref=10):
        ids = tok(text, return_tensors='pt').to(DEVICE).input_ids[0]
        if len(ids) <= n_pref + 1: return float('nan')
        logits = base(ids.unsqueeze(0)).logits[0]
        logp = _F.log_softmax(logits[:-1], dim=-1)
        return float(logp.gather(-1, ids[1:].unsqueeze(-1)).squeeze(-1)[n_pref-1:].mean().item())
    _scores = sorted([(i, _logp(t)) for i, t in enumerate(_CANDIDATES)], key=lambda s: -s[1])
    _keep = sorted([s[0] for s in _scores[:7]])
    _MEM   = [_CANDIDATES[i] for i in _keep]
    _CLEAN = [_CLEAN_POOL[i] for i in _keep]
    json.dump(_MEM,   open(DRIVE / 'pythia_memorized.json', 'w'), indent=2)
    json.dump(_CLEAN, open(DRIVE / 'pythia_clean.json',     'w'), indent=2)
    print(f'  built and saved to {DRIVE}/')

MEM   = json.load(open(DRIVE / 'pythia_memorized.json'))
CLEAN = json.load(open(DRIVE / 'pythia_clean.json'))
assert len(MEM) == len(CLEAN), 'mem/clean must be matched count'
N = len(MEM); ALL = MEM + CLEAN; Y = np.array([1]*N + [0]*N)
print(f'  N memorized = {N}, N clean = {N}')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded EleutherAI/pythia-70m: 70,426,624 params
pythia_memorized.json/pythia_clean.json missing -- running build step (idempotent, ~1 min on T4)...
  built and saved to MIDU/
  N memorized = 7, N clean = 7


## 3. Probe primitives + baseline

In [13]:
@torch.no_grad()
def acts_at_layer(m, texts, layer):
    out = []
    for t in texts:
        ids = tok(t, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        h = m(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
        out.append(h.cpu().float().numpy())
    return np.array(out)

def loo_probe(X, y, C=PROBE_C, seed=42):
    accs = []
    for i in range(N):
        te = np.array([(j == i) or (j == N + i) for j in range(2*N)])
        sc = StandardScaler(); Xtr = sc.fit_transform(X[~te]); Xte = sc.transform(X[te])
        accs.append(LogisticRegression(max_iter=10000, C=C, random_state=seed)
                    .fit(Xtr, y[~te]).score(Xte, y[te]))
    return float(np.mean(accs))

def fit_w(X, y):
    sc = StandardScaler(); Xn = sc.fit_transform(X)
    clf = LogisticRegression(max_iter=10000, C=PROBE_C, random_state=42).fit(Xn, y)
    w = clf.coef_[0] / sc.scale_; return w / (np.linalg.norm(w) + 1e-12)

print('=== BASELINE LOO probe per layer ===')
base.eval(); pre = {}
for L in range(N_HIDDEN):
    pre[L] = loo_probe(acts_at_layer(base, ALL, L), Y)
    print(f'  layer {L}: {pre[L]:.3f}')

=== BASELINE LOO probe per layer ===
  layer 0: 0.643
  layer 1: 0.714
  layer 2: 0.786
  layer 3: 0.857
  layer 4: 0.857
  layer 5: 0.929
  layer 6: 0.929


## 4. PGA training (LoRA on attention + MLP)

In [14]:
# ===== reproducibility patch =====
# Reset RNGs immediately before LoRA init/training so the LoRA A-matrix
# initialization (kaiming_uniform) and DataLoader shuffling start from a
# known state — regardless of how much the random state was advanced by
# upstream cells (model load, data-pool auto-build, probe fitting).
import random
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
random.seed(42); np.random.seed(42)
# For full CUDA determinism (slower), uncomment the next two lines AND
# set CUBLAS_WORKSPACE_CONFIG=":16:8" in cell 1 BEFORE importing torch:
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False
# =================================

lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=2*LORA_R,
                     target_modules=['query_key_value','dense','dense_h_to_4h','dense_4h_to_h'],
                     lora_dropout=0.0, bias='none')
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

def refit_w_layers(m, layers):
    m.eval(); out = {}
    for L in layers:
        X = acts_at_layer(m, ALL, L)
        out[L] = torch.as_tensor(fit_w(X, Y), dtype=torch.float32, device=DEVICE)
    m.train(); return out

def pga_step(m, opt, w_per_layer, layers):
    m.train(); opt.zero_grad()
    align = 0.0; ce = 0.0
    for i in range(N):
        m_ids = tok(MEM[i], return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        c_ids = tok(CLEAN[i], return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        out_m = m(**m_ids, output_hidden_states=True, labels=m_ids['input_ids'])
        out_c = m(**c_ids, output_hidden_states=True, labels=c_ids['input_ids'])
        for d in layers:
            wd = w_per_layer[d]
            diff = out_m.hidden_states[d][0, -1, :] - out_c.hidden_states[d][0, -1, :]
            align = align + (diff @ wd) ** 2
        ce = ce + out_c.loss
    align = align / (N * len(layers)); ce = ce / N
    loss = LAMBDA_ALIGN * align + LAMBDA_CE * ce
    loss.backward(); opt.step()
    return float(align.item()), float(ce.item())

opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
print(f'=== PGA training (LoRA r={LORA_R}, λ_align={LAMBDA_ALIGN}) ===')
t0 = time.time()
w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
for ep in range(1, EPOCHS + 1):
    if ep > 1 and (ep - 1) % REFIT_EVERY == 0:
        w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
    al, ce = pga_step(model, opt, w_per_layer, ALIGN_LAYERS)
    if ep % 25 == 0 or ep == 1:
        model.eval()
        probe_at = loo_probe(acts_at_layer(model, ALL, PROBE_LAYER), Y)
        model.train()
        print(f'ep {ep:3d}  align {al:.4f}  ce {ce:.3f}  '
              f'probe@L{PROBE_LAYER} {probe_at:.3f}  elapsed {time.time()-t0:.0f}s')
model.eval()

trainable params: 786,432 || all params: 71,213,056 || trainable%: 1.1043
=== PGA training (LoRA r=16, λ_align=1.0) ===
ep   1  align 211.9555  ce 4.021  probe@L4 0.857  elapsed 2s
ep  25  align 7.9282  ce 4.476  probe@L4 0.786  elapsed 11s
ep  50  align 4.3249  ce 4.883  probe@L4 0.643  elapsed 22s
ep  75  align 2.6132  ce 4.831  probe@L4 0.571  elapsed 33s
ep 100  align 1.9136  ce 4.587  probe@L4 0.429  elapsed 44s
ep 125  align 1.4000  ce 4.290  probe@L4 0.429  elapsed 54s
ep 150  align 1.0106  ce 4.033  probe@L4 0.429  elapsed 65s
ep 175  align 0.7843  ce 3.762  probe@L4 0.429  elapsed 76s
ep 200  align 0.7068  ce 3.522  probe@L4 0.429  elapsed 87s


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPTNeoXForCausalLM(
      (gpt_neox): GPTNeoXModel(
        (embed_in): Embedding(50304, 512)
        (emb_dropout): Dropout(p=0.0, inplace=False)
        (layers): ModuleList(
          (0-5): 6 x GPTNeoXLayer(
            (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (post_attention_dropout): Dropout(p=0.0, inplace=False)
            (post_mlp_dropout): Dropout(p=0.0, inplace=False)
            (attention): GPTNeoXAttention(
              (query_key_value): lora.Linear(
                (base_layer): Linear(in_features=512, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=512, out_features=16, bias=False)
                )
    

## 5. Post-PGA per-layer probe + save LoRA adapter

In [15]:
print('=== POST-PGA LOO probe per layer ===')
post = {}
for L in range(N_HIDDEN):
    post[L] = loo_probe(acts_at_layer(model, ALL, L), Y)
    print(f'  layer {L}: pre {pre[L]:.3f}  ->  post {post[L]:.3f}  (Δ {post[L]-pre[L]:+.3f})')

json.dump({'model': MODEL_NAME, 'lora_r': LORA_R, 'epochs': EPOCHS,
           'lambda_align': LAMBDA_ALIGN, 'lambda_ce': LAMBDA_CE,
           'pre_per_layer':  {str(k): float(v) for k, v in pre.items()},
           'post_per_layer': {str(k): float(v) for k, v in post.items()},
           'peak_gap_layer': PROBE_LAYER},
          open(ART / 'mldu_e_pga_pythia70m.json', 'w'), indent=2)
print(f'\nsaved per-layer json: {ART / "mldu_e_pga_pythia70m.json"}')

# Save LoRA adapter so a later session can reload without retraining
model.save_pretrained(str(LORA_DIR))
print(f'saved LoRA adapter:  {LORA_DIR}')

=== POST-PGA LOO probe per layer ===
  layer 0: pre 0.643  ->  post 0.643  (Δ +0.000)
  layer 1: pre 0.714  ->  post 0.714  (Δ +0.000)
  layer 2: pre 0.786  ->  post 0.571  (Δ -0.214)
  layer 3: pre 0.857  ->  post 0.286  (Δ -0.571)
  layer 4: pre 0.857  ->  post 0.429  (Δ -0.429)
  layer 5: pre 0.929  ->  post 0.286  (Δ -0.643)
  layer 6: pre 0.929  ->  post 0.143  (Δ -0.786)

saved per-layer json: MIDU/MLDU_E/artifacts/mldu_e_pga_pythia70m.json
saved LoRA adapter:  MIDU/pythia70m_pga_lora


## 6. Held-out probe attack — six adversarial probe variants

Tests whether PGA's collapse holds against probe-shopping at Pythia-70M scale. Same protocol as the toy robustness check (Appendix `app:mldu_e_robustness`).

In [16]:
VARIANTS = [
    ('LR seed=42 C=1.0  (trained-against)', 'lr',  {'random_state': 42,  'C': 1.0}),
    ('LR seed=7  C=1.0  (held-out seed)',   'lr',  {'random_state': 7,   'C': 1.0}),
    ('LR seed=13 C=0.1  (more regularized)','lr',  {'random_state': 13,  'C': 0.1}),
    ('LR seed=99 C=10.0 (less regularized)','lr',  {'random_state': 99,  'C': 10.0}),
    ('MLP[16] seed=42   (nonlinear)',       'mlp', {'random_state': 42,  'hidden_layer_sizes': (16,)}),
    ('MLP[32,16] seed=7 (deeper nonlinear)','mlp', {'random_state': 7,   'hidden_layer_sizes': (32, 16)}),
]

def fit_variant(Xtr, ytr, Xte, yte, kind, kw):
    sc = StandardScaler(); Xtrn = sc.fit_transform(Xtr); Xten = sc.transform(Xte)
    if kind == 'lr':
        clf = LogisticRegression(max_iter=10000, **kw).fit(Xtrn, ytr)
    else:
        clf = MLPClassifier(max_iter=3000, early_stopping=False, tol=1e-5, **kw).fit(Xtrn, ytr)
    return float(clf.score(Xten, yte))

def loo_variant(Xd, kind, kw):
    accs = []
    for i in range(N):
        te = np.array([(j == i) or (j == N + i) for j in range(2*N)])
        accs.append(fit_variant(Xd[~te], Y[~te], Xd[te], Y[te], kind, kw))
    return float(np.mean(accs))

# Cache activations once per layer
print('Collecting PGA-edited activations per layer...')
acts_post = {L: acts_at_layer(model, ALL, L) for L in range(N_HIDDEN)}

header = '\n' + f'{"variant":<42} ' + ' '.join(f'L{L:>2}' for L in range(N_HIDDEN)) + '   max'
print(header)
print('-' * len(header))
robustness = {}
for label, kind, kw in VARIANTS:
    per_layer = [loo_variant(acts_post[L], kind, kw) for L in range(N_HIDDEN)]
    mp = max(per_layer)
    robustness[label] = {'per_layer': per_layer, 'max': mp}
    parts = ' '.join(f'{a:>3.2f}' for a in per_layer)
    print(f'{label:<42} {parts}  {mp:.3f}')

worst = max(r['max'] for r in robustness.values())
worst_var = max(robustness.items(), key=lambda kv: kv[1]['max'])[0]
print(f'\nworst-case max probe: {worst:.3f}  (variant: {worst_var})')
print(f'target: max probe ≤ 0.72')
if worst <= 0.72:
    print('PASS — PGA collapse holds against probe-shopping at Pythia-70M scale.')
else:
    print('PARTIAL — some probe variants exceed floor; report scope as linear-only.')

json.dump({
    'model': MODEL_NAME, 'n_variants': len(VARIANTS),
    'variants': {k: {'per_layer': v['per_layer'], 'max': v['max']}
                 for k, v in robustness.items()},
    'worst_max': float(worst), 'worst_variant': worst_var,
    'pass': bool(worst <= 0.72),
    'pre_per_layer':  {str(k): float(v) for k, v in pre.items()},
    'post_per_layer': {str(k): float(v) for k, v in post.items()},
}, open(ART / 'mldu_e_pga_pythia70m_robustness.json', 'w'), indent=2)
print(f'\nsaved: {ART / "mldu_e_pga_pythia70m_robustness.json"}')


variant                                    L 0 L 1 L 2 L 3 L 4 L 5 L 6   max
-----------------------------------------------------------------------------
LR seed=42 C=1.0  (trained-against)        0.64 0.71 0.57 0.29 0.43 0.29 0.14  0.714
LR seed=7  C=1.0  (held-out seed)          0.64 0.71 0.57 0.29 0.43 0.29 0.14  0.714
LR seed=13 C=0.1  (more regularized)       0.64 0.71 0.57 0.29 0.43 0.29 0.14  0.714
LR seed=99 C=10.0 (less regularized)       0.79 0.71 0.57 0.29 0.43 0.29 0.07  0.786
MLP[16] seed=42   (nonlinear)              0.64 0.71 0.57 0.29 0.43 0.43 0.14  0.714
MLP[32,16] seed=7 (deeper nonlinear)       0.64 0.79 0.71 0.36 0.43 0.50 0.36  0.786

worst-case max probe: 0.786  (variant: LR seed=99 C=10.0 (less regularized))
target: max probe ≤ 0.72
PARTIAL — some probe variants exceed floor; report scope as linear-only.

saved: MIDU/MLDU_E/artifacts/mldu_e_pga_pythia70m_robustness.json


In [17]:
# === Generate robustness figure (toy + Pythia-70M) ===
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = Path('/content/drive/MyDrive/MIDU')
except Exception:
    BASE = Path('.')
ART = BASE / 'MLDU_E' / 'artifacts'
FIG = BASE / 'MLDU_E' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)

# JSONs embedded inline so this cell is self-contained on Kaggle/Colab/local
# (the originals live at MLDU-main/results/mldu_e/{mldu_e_pga_robustness, mldu_e_pga_pythia70m_robustness}.json)
_EMBED_TOY = {
  "best_lambda": 0.1,
  "variants": {
    "LR seed=42 C=1.0  (trained-against)": {"per_depth": [0.65, 0.5277777777777778, 0.2944444444444445, 0.17777777777777778, 0.17222222222222222], "max": 0.65},
    "LR seed=7  C=1.0  (held-out seed)":   {"per_depth": [0.65, 0.5277777777777778, 0.2944444444444445, 0.17777777777777778, 0.17222222222222222], "max": 0.65},
    "LR seed=13 C=0.1  (more regularized)":{"per_depth": [0.65, 0.5722222222222222, 0.35555555555555557, 0.2666666666666667, 0.2611111111111111], "max": 0.65},
    "LR seed=99 C=10.0 (less regularized)":{"per_depth": [0.65, 0.5611111111111112, 0.27222222222222225, 0.16666666666666666, 0.15], "max": 0.65},
    "MLP[16] seed=42   (nonlinear)":       {"per_depth": [0.6611111111111112, 0.6111111111111112, 0.4555555555555556, 0.3833333333333333, 0.35555555555555557], "max": 0.6611111111111112},
    "MLP[32,16] seed=7 (deeper nonlinear)":{"per_depth": [0.6611111111111112, 0.5888888888888889, 0.5055555555555555, 0.4, 0.3833333333333333], "max": 0.6611111111111112}
  },
  "worst_max_across_variants": 0.6611111111111112,
  "worst_variant": "MLP[16] seed=42   (nonlinear)",
  "pass": True
}

_EMBED_PYTHIA = {
  "model": "EleutherAI/pythia-70m",
  "n_variants": 6,
  "variants": {
    "LR seed=42 C=1.0  (trained-against)": {"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.14285714285714285], "max": 0.7142857142857143},
    "LR seed=7  C=1.0  (held-out seed)":   {"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.14285714285714285], "max": 0.7142857142857143},
    "LR seed=13 C=0.1  (more regularized)":{"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.14285714285714285], "max": 0.7142857142857143},
    "LR seed=99 C=10.0 (less regularized)":{"per_layer": [0.7857142857142857, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.07142857142857142], "max": 0.7857142857142857},
    "MLP[16] seed=42   (nonlinear)":       {"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.42857142857142855, 0.14285714285714285], "max": 0.7142857142857143},
    "MLP[32,16] seed=7 (deeper nonlinear)":{"per_layer": [0.6428571428571429, 0.7857142857142857, 0.7142857142857143, 0.35714285714285715, 0.42857142857142855, 0.5, 0.35714285714285715], "max": 0.7857142857142857}
  },
  "worst_max": 0.7857142857142857,
  "worst_variant": "LR seed=99 C=10.0 (less regularized)",
  "pass": False,
  "pre_per_layer":  {"0": 0.6428571428571429, "1": 0.7142857142857143, "2": 0.7857142857142857, "3": 0.8571428571428571, "4": 0.8571428571428571, "5": 0.9285714285714286, "6": 0.9285714285714286},
  "post_per_layer": {"0": 0.6428571428571429, "1": 0.7142857142857143, "2": 0.5714285714285714, "3": 0.2857142857142857, "4": 0.42857142857142855, "5": 0.2857142857142857, "6": 0.14285714285714285}
}

def _load_robust(name, embedded):
    """Load from disk if available (latest local results), else fall back to embedded copy."""
    candidates = [
        ART / name,
        Path('../results/mldu_e') / name,
        Path('./results/mldu_e') / name,
        Path('/kaggle/input/mldu/results/mldu_e') / name,
    ]
    for p in candidates:
        if p.exists():
            print(f'  loaded from disk: {p}')
            return json.load(open(p))
    print(f'  using embedded copy of {name} (no disk file found)')
    return embedded

toy_robust    = _load_robust('mldu_e_pga_robustness.json',          _EMBED_TOY)
pythia_robust = _load_robust('mldu_e_pga_pythia70m_robustness.json', _EMBED_PYTHIA)

def get_variant_curves(robust_dict, key='variants'):
    """Returns ordered (label, per_layer) tuples."""
    out = []
    src = robust_dict[key]
    for label, info in src.items():
        per = info.get('per_depth') or info.get('per_layer')
        out.append((label, per))
    return out

toy_curves    = get_variant_curves(toy_robust)
pythia_curves = get_variant_curves(pythia_robust)

# Style: 4 LR variants in shades of blue, 2 MLP variants in red/orange
COLORS = {
    'LR seed=42 C=1.0':  '#1f77b4',
    'LR seed=7':         '#4a90d9',
    'LR seed=13 C=0.1':  '#7eb6e8',
    'LR seed=99 C=10.0': '#aed6f1',
    'MLP[16]':           '#d62728',
    'MLP[32,16]':        '#ff8c42',
}

def color_for(label):
    for k, c in COLORS.items():
        if k in label: return c
    return '#888888'

def short(label):
    s = label.split('  ')[0].strip()
    return s

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Panel 1: toy (5 depths)
ax = axes[0]
xs_toy = list(range(5))
for label, per in toy_curves:
    ax.plot(xs_toy, per, marker='o', lw=2, ms=6, color=color_for(label), label=short(label))
ax.axhline(0.72, ls='--', color='#222222', lw=1.4, alpha=0.7, label='floor target (0.72)')
ax.axhline(0.50, ls=':',  color='#222222', lw=1.0, alpha=0.5, label='random (0.50)')
ax.axvspan(0.5, 4.5, color='#2874a6', alpha=0.06, label='mem-relevant')
ax.set_xticks(xs_toy); ax.set_xticklabels([f'd{d}' for d in xs_toy])
ax.set_ylim(0.0, 1.05); ax.set_xlabel('residual depth'); ax.set_ylabel('LOO probe accuracy')
ax.set_title('Toy (0.8M params, 4 layers + embed)\nworst-case max probe: 0.661')
ax.legend(loc='lower left', fontsize=8, ncol=1, framealpha=0.9)
ax.grid(alpha=0.3)

# Panel 2: Pythia-70M (7 layers)
ax2 = axes[1]
xs_p = list(range(7))
for label, per in pythia_curves:
    ax2.plot(xs_p, per, marker='o', lw=2, ms=6, color=color_for(label), label=short(label))
ax2.axhline(0.72, ls='--', color='#222222', lw=1.4, alpha=0.7, label='floor target (0.72)')
ax2.axhline(0.50, ls=':',  color='#222222', lw=1.0, alpha=0.5, label='random (0.50)')
ax2.axvspan(1.5, 6.5, color='#2874a6', alpha=0.06, label='mem-relevant (L2-L6)')
ax2.annotate('LR C=10\nleak at L0', xy=(0, 0.79), xytext=(0.3, 0.95),
             fontsize=8, ha='left', arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.2))
ax2.annotate('MLP[32,16]\nleak at L1', xy=(1, 0.79), xytext=(1.6, 0.95),
             fontsize=8, ha='left', arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.2))
ax2.set_xticks(xs_p); ax2.set_xticklabels([f'L{L}' for L in xs_p])
ax2.set_ylim(0.0, 1.05); ax2.set_xlabel('residual layer'); ax2.set_ylabel('LOO probe accuracy')
ax2.set_title('Pythia-70M (70M params, 6 layers + embed)\nworst-case overall: 0.786 - at L0/L1 token-identity layers'
              '\nworst-case at mem-relevant layers (L2-L6): 0.71')
ax2.legend(loc='lower left', fontsize=8, ncol=1, framealpha=0.9)
ax2.grid(alpha=0.3)

plt.suptitle('PGA robustness: 6 adversarial probe variants per architecture', fontsize=12, y=1.02)
plt.tight_layout()

OUT = FIG / 'mldu_e_pga_robustness.png'
plt.savefig(OUT, dpi=300, bbox_inches='tight')
plt.show()
print(f'saved: {OUT}')

  using embedded copy of mldu_e_pga_robustness.json (no disk file found)
  using embedded copy of mldu_e_pga_pythia70m_robustness.json (no disk file found)
saved: MLDU_E/figures/mldu_e_pga_robustness.png
